In [1]:
import pandas as pd
import json
import os
import re
from pathlib import Path
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # ensure consistency

In [2]:
def detect_language(text: str) -> str:
    try:
        return detect(text)
    except:
        return "unknown"
        
def get_text(srt_filepath):
    """
    Reads an .srt file and returns the full subtitle text as a single string.
    Handles multi-line subtitles properly.
    """
    try:
        with open(srt_filepath, 'r', encoding='utf-8') as f:
            content = f.read()

        # Split into subtitle blocks (separated by blank lines)
        blocks = re.split(r'\n\s*\n', content.strip())
        all_text = []

        for block in blocks:
            block = re.sub(r'^\d+\s*\n', '', block)
            # Remove timestamp lines
            block = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', block)
            # Remove extra whitespace
            block = block.strip()
            if block:
                all_text.append(block.replace('\n', ' '))  # merge multi-line captions

        # Join all subtitle blocks into one paragraph
        return ' '.join(all_text)

    except FileNotFoundError:
        return "Error: SRT file not found."
    except Exception as e:
        return f"An error occurred: {e}"

def to_english_text(x):
    """Accepts either a string or a dict from get_text(); returns best English string."""
    if x is None:
        return None
    if isinstance(x, dict):
        return x.get("translated_text") or x.get("text") or ""
    return x

BEHAVIOR_VOCAB_ORDER = [
    "Absence or Avoidance of Eye Contact",
    "Aggressive Behavior",
    "Hyper- or Hyporeactivity to Sensory Input",
    "Non-Responsiveness to Verbal Interaction",
    "Non-Typical Language",
    "Object Lining-Up",
    "Self-Hitting or Self-Injurious Behavior",
    "Self-Spinning or Spinning Objects",
    "Upper Limb Stereotypies",
]
NEGATIVE_CLASS = "Background"

INSTRUCTION_HEADER = (
    "Given the sequence of images below, with Audio Caption and Speech Transcription, "
    "indicate which, if any, of the following categories of autism-related behaviors are present: "
    "{Absence or Avoidance of Eye Contact; Aggressive Behavior; Hyper- or Hyporeactivity to Sensory Input; "
    "Non-Responsiveness to Verbal Interaction; Non-Typical Language; Object Lining-Up; "
    "Self-Hitting or Self-Injurious Behavior; Self-Spinning or Spinning Objects; Upper Limb Stereotypies}. "
    "If none are present, answer: Background."
)

In [ ]:
data_dir = Path("../dataset/output")
csv_df = pd.read_csv("../dataset/csvs/dataset.csv")
id_col = "Video_ID"
inference_df = pd.DataFrame()

records = []

for folder_path in (p for p in data_dir.iterdir() if p.is_dir()):
    video_id = folder_path.name

    if csv_df.loc[csv_df[id_col] == video_id].empty:
        print(f"[WARN] No ground-truth for {video_id}")
        continue

    ground_truth_df=csv_df[csv_df[id_col]==video_id] 
    for _, row in ground_truth_df.iterrows(): 
        nonzero_cols = row[row != 0].drop(labels=[id_col]).index.tolist() 

    files = list(folder_path.iterdir())

    # Pick first image, caption (.srt), transcript (.txt) by pattern
    image_path = next((p for p in files if p.suffix.lower() in {".jpg", ".jpeg", ".png"}), None)
    srt_path = next((p for p in files if p.name.lower().endswith("_srt.srt")), None)
    transcript_path = next((p for p in files if p.name.lower().endswith("_transcript.txt")), None)

    # Read once (cached)
    raw_caption = read_cached(srt_path, get_text) if srt_path else None
    raw_transcript = read_cached(transcript_path, get_text) if transcript_path else None

    # Convert to final English strings (handles dict-or-string returns)
    caption_txt = to_english_text(raw_caption)
    transcript_txt = to_english_text(raw_transcript)

    # Only add rows that have at least an image and some text (tweak logic as needed)
    if image_path is None:
        print(f"[WARN] No image found for {video_id}")
        continue
    if caption_txt is None and transcript_txt is None:
        print(f"[WARN] No caption/transcript for {video_id}")
        continue
    lang_txt = detect_language(caption_txt)
    if lang_txt.lower()!='en':
        en_caption_txt=to_english_text(caption_txt)
        en_transcript_txt=to_english_text(transcript_txt)

    records.append({
        "id": video_id,
        "path": str(image_path),
        "caption": caption_txt or "",
        "transcript": transcript_txt or "",
        "detected_lang": lang_txt,
        "en_caption_txt": caption_txt,
        "en_transcript_txt": transcript_txt,
        "behavior_txt": nonzero_cols
    })

raw_inference_df = pd.DataFrame.from_records(records, columns=[
    "id", "path", "transcript", "caption", "detected_lang", "en_caption_txt", "en_transcript_txt", "behavior_txt"
    ])

inference_df = raw_inference_df[raw_inference_df['caption']!="None"]

display(inference_df.head())

In [ ]:
def select_text(row, caption_first=True):
    """
    Prefer English fields if present; fall back to originals.
    Returns (caption_text, transcript_text).
    """
    cap = row.get("en_caption_txt") or row.get("caption") or ""
    trans = row.get("en_transcript_txt") or row.get("transcript") or ""
    return (cap, trans)

def format_labels(labels):
    """
    labels: list[str] from row['behavior_txt'].
    Ensure deterministic order and join with '; '.
    If empty, return NEGATIVE_CLASS.
    """
    if not labels:
        return NEGATIVE_CLASS
    order_map = {k: i for i, k in enumerate(BEHAVIOR_VOCAB_ORDER)}
    labels_sorted = sorted(set(labels), key=lambda x: order_map.get(x, 9999))
    return "; ".join(labels_sorted)

def build_human_value(caption_text, transcript_text):
    """
    Compose the human turn with caption/transcription + <image>.
    """
    caption_text = caption_text.strip()
    transcript_text = transcript_text.strip()
    return (
        f"{INSTRUCTION_HEADER}\n\n"
        f"Audio Caption: {caption_text}\n"
        f"Speech Transcription: {transcript_text}\n\n"
        f"<image>"
    )

def row_to_llava_example(row):
    vid = str(row["id"])
    image_path = str(row["path"])
    behaviors = row.get("behavior_txt") or []
    if isinstance(behaviors, str):
        # if it slipped in as a string, try to parse simple repr like "['A','B']" or split
        try:
            import ast
            parsed = ast.literal_eval(behaviors)
            if isinstance(parsed, list):
                behaviors = parsed
        except Exception:
            behaviors = [b.strip() for b in behaviors.split(";") if b.strip()]

    cap_txt, trans_txt = select_text(row)

    human_val = build_human_value(cap_txt, trans_txt)
    gpt_val = format_labels(behaviors)

    ex = {
        "id": vid,
        "image": image_path,
        "conversations": [
            {"from": "human", "value": human_val},
            {"from": "gpt", "value": gpt_val},
        ]
    }
    return ex

# Build the full JSONL
OUTPUT_JSONL = "../dataset/instruction/llava_instruct.jsonl"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f_out:
    for _, row in inference_df.iterrows():
        ex = row_to_llava_example(row)      # builds the dict
        f_out.write(json.dumps(ex, ensure_ascii=False) + "\n")
        
print(f"Wrote JSONL to {OUTPUT_JSONL}")
examples = [json.loads(line) for line in open(OUTPUT_JSONL, "r", encoding="utf-8")]
print(examples[0])

## Prepare data for Llama-Factory fine tune

In [ ]:
import os
import shutil
from pathlib import Path

# Base directory containing folders with images
base_dir = Path("../dataset/output")
# Destination folder for all images
dest_dir = base_dir / "asd_demo_data"
# dest_dir = Path("/home/anirban/projects/AV-ASD/dataset/asd_demo_data")


# Create the destination directory if it doesn't exist
# dest_dir.mkdir(parents=True, exist_ok=True)

# Traverse all subdirectories under base_dir
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.lower().endswith(".jpg"):
            src_path = Path(root) / file
            dest_path = dest_dir / file

            # Handle duplicate filenames by prefixing parent folder name
            if dest_path.exists():
                parent_name = Path(root).name
                dest_path = dest_dir / f"{parent_name}_{file}"

            shutil.copy2(src_path, dest_path)

print("✅ All .jpg files copied to ../dataset/output/asd_demo_data/")

In [ ]:
# Update image file location in the json
import json

input_file = "../dataset/instruction/asd_demo.jsonl"
output_file = "../dataset/instruction/asd_demo.json"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        data = json.loads(line)

        # Get filename only (last part)
        filename = os.path.basename(data["image"])

        # Build new path
        data["image"] = f"asd_demo_data/{filename}"

        # Write back to JSONL
        outfile.write(json.dumps(data) + "\n")

print("✅ Updated file saved as:", output_file)


## Convert ndjson to normal json

In [ ]:
# Convert the ndjson to normal json
def ndjson_to_array(in_file, out_file):
    data = []
    with open(in_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

ndjson_to_array("../dataset/instruction/asd_demo.json", "../dataset/instruction/asd_demo.json")

## Change "image" to "images" in json

In [ ]:
import json
with open("/home/anirban/LLaMA-Factory/data/asd_demo.json","r") as f:
    data = json.load(f)
for ex in data:
    if "image" in ex and "images" not in ex:
        ex["images"] = [ex["image"]]
        del ex["image"]
with open("/home/anirban/LLaMA-Factory/data/asd_demo.json","w") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

## Check for mismatches

In [ ]:
import json, re
from pathlib import Path

p = Path("/home/anirban/LLaMA-Factory/data/asd_demo.json")
data = json.loads(p.read_text())

bad = []
for i, ex in enumerate(data):
    msgs = ex.get("conversations", [])
    # count <image> tokens across human messages
    text = "\n".join(m["value"] for m in msgs if m.get("from") == "human")
    n_tokens = len(re.findall(r"<image>", text))
    imgs = ex.get("images", [])
    if isinstance(imgs, str):
        imgs = [imgs]
    n_imgs = len(imgs)
    if n_tokens != n_imgs:
        bad.append((i, n_tokens, n_imgs, imgs))

print(f"Checked {len(data)} samples. Mismatches: {len(bad)}")
for i, nt, ni, imgs in bad[:20]:
    print(f"idx={i} <image>={nt} images={ni} first_img={imgs[0] if imgs else None}")


In [ ]:
import json, re, os
from pathlib import Path

src = Path("/home/anirban/LLaMA-Factory/data/asd_demo.json")
dst = Path("/home/anirban/LLaMA-Factory/data/asd_demo_new.json")

data = json.loads(src.read_text())
clean = []
bad   = []

for ex in data:
    # normalize images to a list
    imgs = ex.get("images", [])
    if isinstance(imgs, str):
        imgs = [imgs]
    # count <image> tokens in human messages
    txt = "\n".join(m["value"] for m in ex.get("conversations", []) if m.get("from")=="human")
#     n_tok = len(re.findall(r"<image>", txt))
#     # keep only simple, 1-image samples with exactly one token
#     if n_tok == 1 and len(imgs) == 1 and os.path.exists(os.path.join("data", imgs[0])):
#         clean.append(ex)
#     else:
#         bad.append({"reason": f"<image>={n_tok}, images={len(imgs)}", "id": ex.get("id")})

# print(f"Kept {len(clean)} / {len(data)}; dropped {len(bad)} mismatched.")
# dst.write_text(json.dumps(clean, ensure_ascii=False, indent=2))

## Copy images and prep json

In [ ]:
#!/usr/bin/env python3
import json
import shutil
import time
from pathlib import Path

# === PATHS (edit if needed) ===
SRC_JSON = Path("/home/anirban/projects/AV-ASD/dataset/instruction/asd_demo.json")
IMG_DIR  = Path("/home/anirban/projects/AV-ASD/dataset/asd_demo_data")

# Overwrite the source JSON in-place (with backup) after cleaning
OVERWRITE_JSON = True

# Safety: move removed images here instead of permanent delete
TRASH_DIR = IMG_DIR / "_trash"

# Treat these as valid image extensions
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: Path, data):
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def list_image_files(dirpath: Path):
    return [p for p in dirpath.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]


if not SRC_JSON.exists():
    raise SystemExit(f"JSON not found: {SRC_JSON}")
if not IMG_DIR.exists():
    raise SystemExit(f"Image directory not found: {IMG_DIR}")

data = load_json(SRC_JSON)

# ---- Pass 1: drop JSON entries whose referenced images are missing in IMG_DIR
kept_entries = []
removed_entries = []   # (id, missing_basenames)
referenced_basenames = set()

for ex in data:
    # Collect image references from "images" (list) or "image" (string)
    imgs = []
    if "images" in ex and isinstance(ex["images"], list):
        imgs = ex["images"]
    elif "image" in ex and isinstance(ex["image"], str):
        imgs = [ex["image"]]
    else:
        removed_entries.append((ex.get("id"), ["<no image field>"]))
        continue

    # Convert to basenames
    basenames = [Path(p).name for p in imgs if isinstance(p, str)]
    if not basenames:
        removed_entries.append((ex.get("id"), ["<empty image list>"]))
        continue

    # Check that ALL referenced basenames exist in IMG_DIR
    missing = [b for b in basenames if not (IMG_DIR / b).exists()]
    if missing:
        removed_entries.append((ex.get("id"), missing))
        continue

    # Keep entry and record basenames it references
    kept_entries.append(ex)
    referenced_basenames.update(basenames)

# ---- Pass 2: remove image files in IMG_DIR that are not referenced by any kept JSON entry
TRASH_DIR.mkdir(parents=True, exist_ok=True)
all_files = list_image_files(IMG_DIR)
orphan_images = [p for p in all_files if p.name not in referenced_basenames]

for p in orphan_images:
    # Move to trash instead of deleting permanently
    target = TRASH_DIR / p.name
    # Avoid overwriting if a same-named file already in trash
    if target.exists():
        stem, ext = p.stem, p.suffix
        target = TRASH_DIR / f"{stem}__{int(time.time())}{ext}"
    shutil.move(str(p), str(target))

# ---- Write cleaned JSON back (with backup) OR print results
if OVERWRITE_JSON:
    # Backup original
    ts = time.strftime("%Y%m%d-%H%M%S")
    backup = SRC_JSON.with_suffix(SRC_JSON.suffix + f".bak.{ts}")
    shutil.copy2(SRC_JSON, backup)
    save_json(SRC_JSON, kept_entries)
    out_where = str(SRC_JSON)
else:
    out_path = SRC_JSON.with_name(SRC_JSON.stem + "_clean.json")
    save_json(out_path, kept_entries)
    out_where = str(out_path)

# ---- Summary
print("✅ Sync complete.")
print(f"- Kept JSON entries: {len(kept_entries)}")
print(f"- Removed JSON entries: {len(removed_entries)}")
if removed_entries:
    print("  • First 20 removed (id -> missing):")
    for rid, miss in removed_entries[:20]:
        print(f"    - {rid} -> {', '.join(miss)}")
    if len(removed_entries) > 20:
        print(f"    ... and {len(removed_entries) - 20} more")

print(f"- Orphan images moved to trash: {len(orphan_images)}  (trash: {TRASH_DIR})")
print(f"- Cleaned JSON written to: {out_where}")



In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

IMG_DIR = Path("/home/anirban/projects/AV-ASD/dataset/asd_demo_data")
JSON_PATH = Path("/home/anirban/projects/AV-ASD/dataset/instruction/asd_demo.json")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Derive the "image id" from a filename.
# Example: cydR1IuT3a8_10_14_grid_3x3.jpg -> cydR1IuT3a8_10_14
GRID_RE = re.compile(r"^(?P<core>.+?)_grid.*$", re.IGNORECASE)

def image_id_from_filename(name: str) -> str:
    stem = Path(name).stem
    m = GRID_RE.match(stem)
    return m.group("core") if m else stem

# 1) collect image files (ignore subfolders)
files = [p for p in IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
img_ids = {}
dup_img_ids = defaultdict(list)
for p in sorted(files):
    iid = image_id_from_filename(p.name)
    if iid in img_ids:
        dup_img_ids[iid].append(p.name)
    else:
        img_ids[iid] = p.name  # map id -> basename

# 2) load json and index by id
data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
json_by_id = {}
dup_json_ids = defaultdict(list)

for i, ex in enumerate(data):
    jid = ex.get("id")
    if not isinstance(jid, str) or not jid.strip():
        dup_json_ids["<missing id>"].append(i)
        continue
    if jid in json_by_id:
        dup_json_ids[jid].append(i)
    else:
        json_by_id[jid] = ex

# 3) compare both sides
missing_in_json = []   # image id exists in folder but no json entry
mismatched_image_name = []  # id matches, but the json["image"] basename != file basename
for iid, basename in img_ids.items():
    ex = json_by_id.get(iid)
    if ex is None:
        missing_in_json.append((iid, basename))
    else:
        # support either "image" (string) or "images" (list)
        image_field_basenames = []
        if isinstance(ex.get("image"), str):
            image_field_basenames.append(Path(ex["image"]).name)
        if isinstance(ex.get("images"), list):
            image_field_basenames.extend(Path(x).name for x in ex["images"] if isinstance(x, str))

        # If there are declared images, check at least one matches the actual basename
        if image_field_basenames and (basename not in image_field_basenames):
            mismatched_image_name.append(
                (iid, basename, sorted(set(image_field_basenames)))
            )

missing_in_folder = []  # json id exists but image file for that id not present
for jid, ex in json_by_id.items():
    # Does IMG_DIR have a file with this id?
    if jid not in img_ids:
        # try to see what json points to (for context)
        candidates = []
        if isinstance(ex.get("image"), str):
            candidates.append(Path(ex["image"]).name)
        if isinstance(ex.get("images"), list):
            candidates.extend(Path(x).name for x in ex["images"] if isinstance(x, str))
        missing_in_folder.append((jid, sorted(set(candidates))))

# 4) print summary
print("=== Image ↔ JSON ID Consistency Report ===")
print(f"Image files found: {len(img_ids)}")
print(f"JSON entries found: {len(json_by_id)}\n")

if dup_img_ids:
    print(f"Duplicate image IDs (same id from multiple files): {len(dup_img_ids)}")
    for iid, names in list(dup_img_ids.items())[:20]:
        print(f"  - {iid}: {', '.join([img_ids[iid]] + names)}")
    if len(dup_img_ids) > 20:
        print(f"  ... and {len(dup_img_ids)-20} more")
    print()

if dup_json_ids:
    print(f"Duplicate JSON IDs: {len(dup_json_ids)}")
    for jid, idxs in list(dup_json_ids.items())[:20]:
        print(f"  - {jid}: nodes {idxs}")
    if len(dup_json_ids) > 20:
        print(f"  ... and {len(dup_json_ids)-20} more")
    print()

print(f"Images without JSON entry: {len(missing_in_json)}")
for iid, basename in missing_in_json[:20]:
    print(f"  - id={iid}  file={basename}")
if len(missing_in_json) > 20:
    print(f"  ... and {len(missing_in_json)-20} more")
print()

print(f"JSON ids without image file: {len(missing_in_folder)}")
for jid, candidates in missing_in_folder[:20]:
    hint = f"  (json points to: {', '.join(candidates)})" if candidates else ""
    print(f"  - id={jid}{hint}")
if len(missing_in_folder) > 20:
    print(f"  ... and {len(missing_in_folder)-20} more")
print()

print(f"ID match but image basename mismatch: {len(mismatched_image_name)}")
for iid, actual, declared in mismatched_image_name[:20]:
    print(f"  - id={iid}  file={actual}  json_has={declared}")
if len(mismatched_image_name) > 20:
    print(f"  ... and {len(mismatched_image_name)-20} more")
print()

# Exit code hint (optional): non-zero if any issues
issues = any([dup_img_ids, dup_json_ids, missing_in_json, missing_in_folder, mismatched_image_name])
if issues:
    print("❌ Inconsistencies found.")
else:
    print("✅ Perfect 1:1 match between folder images and JSON ids.")



### Pre pad 336

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps

src = Path("/home/anirban/projects/AV-ASD/dataset/asd_demo_data")
dst = Path("/home/anirban/LLaMA-Factory/data/asd_demo_data_pad336")
dst.mkdir(parents=True, exist_ok=True)

for p in sorted(src.iterdir()):
    if p.suffix.lower() not in {".jpg",".jpeg",".png",".webp"} or not p.is_file():
        continue
    im = Image.open(p).convert("RGB")
    im = ImageOps.contain(im, (336,336))         # fit within 336x336
    canvas = Image.new("RGB", (336,336), (0,0,0))
    canvas.paste(im, ((336-im.width)//2, (336-im.height)//2))
    canvas.save(dst / p.name, quality=95)